# Silver: CRM customers
**Source:** `bronze.crm_cust_info`  →  **Target:** `silver.crm_customers`

**What this notebook does:**
- Remove extra spaces
- Turn codes into words ((marital_status) M -> Married, S -> Single ; (Gender) M -> Male, F -> Female )
- Drop rows without customer ID
- **Remove duplicate customers** (keep the newest row)
- Rename columns

## Settings

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window

CATALOG = "workspace"

## Read the Bronze table

In [0]:
df = spark.table(f"{CATALOG}.bronze.crm_cust_info")

## 1. Trim spaces

In [0]:
# Remove extra spaces from every text column
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## 2. Turn codes into readable words

In [0]:
df = (
    df
    .withColumn("cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a"))
    .withColumn("cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a"))
    .withColumn("cst_create_date", F.col("cst_create_date").cast(DateType()))
)

## 3. Drop rows without a customer ID

In [0]:
df = df.filter(F.col("cst_id").isNotNull())

# 4. Remove duplicate customers
Some customers appear more than once. The newest row (latest create date) is the most complete, so we keep only that one.

In [0]:
newest_first = Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc())

df = (
    df.withColumn("row_num", F.row_number().over(newest_first))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)

## 5. Rename columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Write the Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.crm_customers")

## Check it Quickly

In [0]:
result = spark.table(f"{CATALOG}.silver.crm_customers")
print("rows:", result.count(), ", unique customer_id:", result.select("customer_id").distinct().count())
result.display()